### Question 2

### TF-IDF

In [5]:
import pandas as pd
import numpy as np
import re
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from tabulate import tabulate

In [6]:
df = pd.read_csv("customer_complaints_1.csv")

In [8]:
def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    words = [w for w in words if w not in ENGLISH_STOP_WORDS]
    return " ".join(words)


In [13]:
# Apply preprocessing on 'text' column
df['cleaned_text'] = df['text'].apply(preprocess)

In [14]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['cleaned_text'])

In [15]:
k = 3   # you can change cluster number
km = KMeans(n_clusters=k)
km.fit(X)

df['Cluster'] = km.predict(X)

In [17]:
print(df[['text', 'Cluster']].head(10))

                                                text  Cluster
0  I used to love Comcast. Until all these consta...        1
1  I'm so over Comcast! The worst internet provid...        1
2  If I could give them a negative star or no sta...        2
3  I've had the worst experiences so far since in...        0
4  Check your contract when you sign up for Comca...        2
5  Thank God. I am changing to Dish. They gave me...        2
6  I Have been a long time customer and only have...        2
7  There is a malfunction on the DVR manager whic...        0
8  Charges overwhelming. Comcast service rep was ...        1
9  I have had cable, DISH, and U-verse, etc. in t...        2


### Word2Vec

In [2]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from gensim.models import Word2Vec
from tabulate import tabulate

# 1. Load the dataset
df = pd.read_csv('customer_complaints_1.csv')
dataset = df['text'].astype(str).tolist()

# 2. Tokenize the dataset
tokenized_dataset = [doc.lower().split() for doc in dataset]

# 3. Train Word2Vec model 
word2vec_model = Word2Vec(sentences=tokenized_dataset, vector_size=100, 
                          window=5, min_count=1, workers=4)

# 4. Create document embeddings 
X = np.array([
    np.mean([word2vec_model.wv[word] for word in doc.lower().split() if word in word2vec_model.wv], axis=0)
    for doc in dataset
])

# 5. Perform clustering 
k = 3
km = KMeans(n_clusters=k, random_state=42)
km.fit(X)

# 6. Predict clusters 
y_pred = km.predict(X)

# 7. Display Results 
table_data = [["Document Snippet", "Predicted Cluster"]]
table_data.extend([[doc[:60] + "...", cluster] for doc, cluster in zip(dataset, y_pred)])
print(tabulate(table_data, headers="firstrow"))

Document Snippet                                                   Predicted Cluster
---------------------------------------------------------------  -------------------
I used to love Comcast. Until all these constant updates. My...                    1
I'm so over Comcast! The worst internet provider. I'm taking...                    2
If I could give them a negative star or no stars on this rev...                    1
I've had the worst experiences so far since install on 10/4/...                    2
Check your contract when you sign up for Comcast as their ad...                    1
Thank God. I am changing to Dish. They gave me awesome prici...                    2
I Have been a long time customer and only have Xfinity as my...                    1
There is a malfunction on the DVR manager which is preventin...                    0
Charges overwhelming. Comcast service rep was so ignorant an...                    1
I have had cable, DISH, and U-verse, etc. in the past. All a...  

C:\Users\nabie\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
